In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from absl import app, flags, logging
from ml_collections import config_flags
import os
from tqdm.notebook import trange

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering

import matplotlib.patches as patches
from scipy.optimize import curve_fit
import numpy.linalg as la

import plotly.io as pio
pio.renderers.default = "notebook_connected"

import json
from ml_collections import ConfigDict

import plotly.graph_objects as go

from icl.linear.train_linear import train
from icl.linear.lr_config import get_config
from icl.linear.lr_task import *
from icl.linear.linear_utils import *
from icl.linear.train_linear import get_sharded_batch_sampler
from icl.linear import DiscreteMMSE, Ridge
from icl.utils import visualize_attention
from icl.figures.task_vec_viz import *

logging.set_verbosity(logging.INFO)
torch.set_printoptions(precision=3, sci_mode=False)
np.set_printoptions(precision=3, suppress=True)

%load_ext autoreload
%autoreload 2

In [18]:
# View attention map
def view_attn(train_task):
    train_task.batch_size = 1
    demo_data0, demo_target = train_task.sample_from_task(train_task.task_pool[0], step=2)
    attns = get_attn(model, demo_data0, demo_target)
    cap = 100
    attns_capped = {layer_key: tensor[:, :cap, :cap] for layer_key, tensor in attns.items()}
    
    widget = visualize_attention(attns_capped, mode='widget')
    return widget

# Utility function to load model and task sampler
def load_model_and_task(exp_name):
    work_dir = os.path.join("..", "results", "linear")
    exp_dir = os.path.join(work_dir, exp_name)
    config_path = os.path.join(exp_dir, "config.json")
    with open(config_path, "r") as f:
        config_dict = json.load(f)
    
    config = ConfigDict(config_dict)
    checkpoint_path = os.path.join(exp_dir, "checkpoint.pt")
    checkpoint = torch.load(checkpoint_path, map_location=config.device)
    data_type = torch.float
    model = get_model(**config["model"], dtype=data_type)
    model.load_state_dict(checkpoint["model"])
    train_task = get_task(**config["task"])
    return model, train_task

def get_attn_mean_var(train_task, model):
    train_task = get_task(**config["task"])
    train_task.batch_size = 256
    demo_data, demo_target = train_task.sample_from_task(train_task.task_pool[1], step=2)
    attns = get_attn(model, demo_data, demo_target)
    attn_means = {layer_key: tensor[:, :, 1::3].mean(dim=0).norm(dim=(-1,-2)).square().cpu().item() for layer_key, tensor in attns.items()}
    attn_vars = {layer_key: tensor[:, :, 1::3].var(dim=0).sum(dim=(-1,-2)).cpu().item() for layer_key, tensor in attns.items()}
    return attn_vars, attn_means

def check_injection(train_task, model, inject_vectors, layer=1, pos=1, is_diff=False, task_idx=None):
    t0 = pos
    l0 = layer
    train_task.batch_size = 1024
    # print(f"Memory allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
    criterion = nn.MSELoss(reduction='none')
    # tvs_means = task_vectors.mean(dim=-2)
    # tvs_mean_weighted = tvs_means[:, -10:].mean(dim=1)
    
    for k in range(config.task.n_tasks):
        torch.cuda.empty_cache()
        if task_idx is None:
            tid = k
        else:
            tid = task_idx
        
        query_data, query_target = train_task.sample_from_task(train_task.task_pool[tid], step=50)
        
        preds_rand = predict_with_task_vector(
                model=model,
                query_data=query_data[:, :(3*t0+3)],
                query_target=query_target[:, :(3*t0+3)],
                task_vector=inject_vectors[k],
                l=l0,              # same layer
                pad="mapsto",      # same position
                pos=3*t0+1,
                is_diff=is_diff
            )
        with torch.no_grad():
            preds = model(query_data, query_target)

        if is_diff:
            query_target[:, t0] = (query_data[:, t0] @ train_task.task_pool[k]).squeeze(-1)
        
        errors_rand = criterion(preds_rand[:, t0], query_target[:, t0].to(preds_rand.device))  # shape: (batch,)
        errors_base = criterion(preds[:, t0], query_target[:, t0].to(preds_rand.device))
        loss = errors_rand.mean()
        baseline_loss = errors_base.mean()
        
        std_rand = errors_rand.std(unbiased=True)
        std_base = errors_base.std(unbiased=True)
        print(f"{t0}-shot loss w. injected vector: {loss.item():.3f} ({std_rand.item():.3f})")
        print(f"{t0}-shot loss w.o. injected vector: {baseline_loss.item():.3f} ({std_base.item():.3f})")

def batch_cosine_similarity(A: np.ndarray, B: np.ndarray, eps: float = 1e-8):
    dot_product = np.sum(A * B, axis=1)
    norm_A = np.linalg.norm(A, axis=1)
    norm_B = np.linalg.norm(B, axis=1)
    return dot_product / (norm_A * norm_B + eps)

def rolling_mean_l2_deviation(arr: torch.Tensor, window: int):
    B, T, D = arr.shape
    output = torch.empty(B, T - window, device=arr.device)

    for b in range(B):
        for t in range(T - window):
            window_slice = arr[b, t:t+window, :]  # (W, D)
            window_mean = window_slice.mean(dim=0)  # (D,)
            l2_distances = torch.norm(window_slice - window_mean, dim=1)  # (W,)
            output[b, t] = l2_distances.mean()

    return output


def plot_mse_vs_position(model, samplers_eval, bayes_ood, bayes_id, mode, step: int = 1):
    torch.cuda.empty_cache()
    
    _data, _, _target = samplers_eval[mode](step=step)
    _data = _data.squeeze(0)
    _target = _target.squeeze(0)

    with torch.no_grad():
        preds = model(_data, _target)
    loss = ((preds - _target.to(preds.device))**2).mean(dim=0)

    re_preds = bayes_ood(_data, _target)
    re_loss = ((re_preds - _target.to(re_preds.device))**2).mean(dim=0)

    dmmse_preds = bayes_id(_data, _target)
    dmmse_loss = ((dmmse_preds - _target.to(dmmse_preds.device))**2).mean(dim=0)

    # Prepare x-axis and numerical arrays
    t = torch.arange(1, loss.shape[0] + 1).cpu().numpy()
    loss_np = loss.cpu().numpy()
    re_loss_np = re_loss.cpu().numpy()
    dmmse_loss_np = dmmse_loss.cpu().numpy()

    # Absolute prediction differences
    diff_dmmse_np = np.abs(preds.mean(dim=0).cpu().numpy() - dmmse_preds.mean(dim=0).cpu().numpy())
    diff_re_np = np.abs(preds.mean(dim=0).cpu().numpy() - re_preds.mean(dim=0).cpu().numpy())

    # Create figure with all traces, but control visibility through buttons
    fig = go.Figure()

    # MSE Traces (Group 0)
    fig.add_trace(go.Scatter(x=t, y=loss_np, mode="lines+markers", name="Model MSE",
                             line=dict(width=2), marker=dict(size=4),
                             hovertemplate="Position: %{x}<br>MSE: %{y:.6f}<extra>Model</extra>",
                             visible=True))
    fig.add_trace(go.Scatter(x=t, y=re_loss_np, mode="lines+markers", name="Ridge MSE",
                             line=dict(width=2, dash="dash"), marker=dict(size=4),
                             hovertemplate="Position: %{x}<br>MSE: %{y:.6f}<extra>Ridge</extra>",
                             visible=True))
    fig.add_trace(go.Scatter(x=t, y=dmmse_loss_np, mode="lines+markers", name="dMMSE MSE",
                             line=dict(width=2, dash="dot"), marker=dict(size=4),
                             hovertemplate="Position: %{x}<br>MSE: %{y:.6f}<extra>dMMSE</extra>",
                             visible=True))

    # Difference Traces (Group 1)
    fig.add_trace(go.Scatter(x=t, y=diff_dmmse_np, mode="lines+markers", name="dMMSE Δ",
                             line=dict(width=2), marker=dict(size=4),
                             hovertemplate="Position: %{x}<br>Δ: %{y:.6f}<extra>dMMSE</extra>",
                             visible=False))
    fig.add_trace(go.Scatter(x=t, y=diff_re_np, mode="lines+markers", name="Ridge Δ",
                             line=dict(width=2, dash="dash"), marker=dict(size=4),
                             hovertemplate="Position: %{x}<br>Δ: %{y:.6f}<extra>Ridge</extra>",
                             visible=False))

    # Dropdown menu
    fig.update_layout(
        updatemenus=[
            dict(
                type="dropdown",
                direction="down",
                showactive=True,
                x=0.25,
                y=1.15,
                buttons=list([
                    dict(label="MSE vs Position",
                         method="update",
                         args=[{"visible": [True, True, True, False, False]},
                               {"title": {"text": "Mean Squared Error vs Position"},
                                "xaxis": {"title": "Position"},
                                "yaxis": {"title": "MSE"}}]),
                    dict(label="Prediction Difference",
                         method="update",
                         args=[{"visible": [False, False, False, True, True]},
                               {"title": {"text": "Prediction Difference vs Position"},
                                "xaxis": {"title": "Position"},
                                "yaxis": {"title": "Absolute Difference"}}])
                ]),
            )
        ]
    )

    # Layout and styling
    fig.update_layout(
        title="Mean Squared Error vs Position",
        xaxis_title="Position",
        yaxis_title="MSE",
        template="plotly_white",
        legend=dict(title="Legend", itemsizing="constant"),
        height=500,
        width=800
    )
    fig.update_xaxes(showgrid=True, gridwidth=0.5, gridcolor='LightGray')
    fig.update_yaxes(showgrid=True, gridwidth=0.5, gridcolor='LightGray')

    fig.show()


def plot_all_relative_errors(train_task, task_vectors, get_dmmse_posterior):
    """
    Plots relative error curves for all k:
        rel_error_k(t) = ||approx_vec - true_vec|| / ||true_vec||
    where approx_vec = E_posterior[lambda] @ task_vectors[:-1, -1]

    Args:
        train_task: input to get_dmmse_posterior (custom format)
        task_vectors: Tensor of shape (num_tasks, seq_len, d)
        get_dmmse_posterior: function(train_task, k) -> (posterior, xs)
    """
    num_tasks, seq_len, _ = task_vectors.shape

    fig = go.Figure()
    x_vals = list(range(seq_len))

    for k in range(num_tasks):
        posterior, xs = get_dmmse_posterior(train_task, k)  # shape: (num_samples, num_tasks)
        bayes_lambdas = posterior.mean(dim=0)               # shape: (num_tasks,)
        approx_vecs = bayes_lambdas[:-1] @ task_vectors[:, -1]  # shape: (d,)
        true_vecs = task_vectors[k]                         # shape: (seq_len, d)

        # Relative error at each t
        rel_error = (approx_vecs - true_vecs).norm(dim=-1) / true_vecs.norm(dim=-1)

        fig.add_trace(go.Scatter(
            x=x_vals,
            y=rel_error.cpu().numpy(),
            mode='lines+markers',
            name=f"k = {k}",
            hovertemplate="t: %{x}<br>Rel. error: %{y:.4f}<extra>k = " + str(k) + "</extra>"
        ))

    fig.update_layout(
        title="Relative Error for Posterior-Weighted Task Vector",
        xaxis_title="t (position)",
        yaxis_title="Relative Error",
        height=500,
        legend_title="Task Index k"
    )

    fig.show()

### $p=0$

In [4]:
# A lazy way to get results from a certain configuration.
# Most experiments share the same hyper parameters except `n_task`, so just change `n_task` and try train it
# If it is already trained then it will show where the results are saved.
# Otherwise, it will run the training procedure.

config = get_config()
# config = u.filter_config(config)
config.task.n_tasks = 4
config.task.p_ood = 0
model, log = train(config)

train_6506dc0f94cd20b46b918868af4e7994 already completed
Loaded model from ../results/linear/train_6506dc0f94cd20b46b918868af4e7994/checkpoint.pt


In [5]:
model, train_task = load_model_and_task("train_6506dc0f94cd20b46b918868af4e7994")
model = model.to(config.device)

In [6]:
train_task.batch_size = 2048
hiddens, xdata = compute_task_vectors(config, model, train_task, layer_index=3)

  0%|          | 0/4 [00:00<?, ?it/s]

In [7]:
global_mean = hiddens.mean(dim=(0,2))
task_vectors = hiddens - global_mean.unsqueeze(0).unsqueeze(2)
task_vectors = task_vectors.mean(dim=-2)
token_vectors = hiddens - task_vectors.unsqueeze(-2) - global_mean.unsqueeze(0).unsqueeze(2)

In [19]:
task_idx = 0
inject_vectors = task_vectors[:,-1] - task_vectors[task_idx:(task_idx+1), -1]
check_injection(train_task, model, inject_vectors, layer=3, pos=20, is_diff=True, task_idx=task_idx)

20-shot loss w. injected vector: 0.001 (0.003)
20-shot loss w.o. injected vector: 0.001 (0.003)
20-shot loss w. injected vector: 0.007 (0.045)
20-shot loss w.o. injected vector: 2.185 (3.153)
20-shot loss w. injected vector: 0.266 (0.400)
20-shot loss w.o. injected vector: 6.865 (10.250)
20-shot loss w. injected vector: 0.004 (0.018)
20-shot loss w.o. injected vector: 1.845 (2.476)


In [169]:
def pairwise_cosine_similarity(X):
    X_norm = F.normalize(X, p=2, dim=1)  # Normalize each row to unit norm
    sim_matrix = X_norm @ X_norm.T       # Dot product between rows
    return sim_matrix
    
pairwise_cosine_similarity(task_vectors[:,-1])

tensor([[ 1.000, -0.260, -0.380, -0.260],
        [-0.260,  1.000, -0.415, -0.291],
        [-0.380, -0.415,  1.000, -0.377],
        [-0.260, -0.291, -0.377,  1.000]])

In [171]:
pairwise_cosine_similarity(train_task.task_pool.squeeze(-1))

tensor([[ 1.000,  0.639, -0.048,  0.682],
        [ 0.639,  1.000, -0.556,  0.561],
        [-0.048, -0.556,  1.000, -0.180],
        [ 0.682,  0.561, -0.180,  1.000]])

In [116]:
plotly_line_plot(global_mean.norm(dim=-1))

In [117]:
plotly_line_plot((global_mean - global_mean[-1:]).norm(dim=-1))

In [118]:
plot_task_vector_variance_with_fit(hiddens)

In [119]:
tvs_diff_means = plot_task_vector_differences(hiddens)

In [120]:
plot_pairwise_task_vector_variance(hiddens)

In [47]:
samplers_eval = {
        get_task_name(task): get_sharded_batch_sampler(task, True)
        for task in train_task.get_default_eval_tasks(**config["eval"])
    }

RE = Ridge(config.task.noise_scale**2 / config.task.task_scale**2)
dMMSE = DiscreteMMSE(config.task.noise_scale, train_task.task_pool)

plot_mse_vs_position(model, samplers_eval, RE, dMMSE, mode="Pretrain", step=1)

In [48]:
plot_mse_vs_position(model, samplers_eval, RE, dMMSE, mode="Latent", step=1)

In [121]:
lambdas, r2_scores = estimate_lambda_with_r2(task_vectors[:,-1], task_vectors)
plot_lambdas(lambdas)

In [149]:
plot_all_relative_errors(train_task, task_vectors, get_dmmse_posterior)

In [73]:
eval_config = config
eval_config.task.n_tasks = 20
eval_task = get_task(**eval_config["task"])
eval_hiddens, eval_xdata = compute_task_vectors(eval_config, model, eval_task, layer_index=2)
eval_global_mean = eval_hiddens.mean(dim=(0,2))
eval_task_vectors = eval_hiddens - eval_global_mean.unsqueeze(0).unsqueeze(2)
eval_task_vectors = eval_task_vectors.mean(dim=-2)
eval_token_vectors = eval_hiddens - eval_task_vectors.unsqueeze(-2) - eval_global_mean.unsqueeze(0).unsqueeze(2)
lambdas, r2_scores = estimate_lambda_with_r2(task_vectors[:,-1], eval_task_vectors)
plot_lambdas(lambdas)

  0%|          | 0/20 [00:00<?, ?it/s]

In [37]:
X = task_vectors[:,-1].numpy()
eigenvalues = np.linalg.eigvals(X @ X.T)
print("Eigenvalues of X^T X:", eigenvalues)

Eigenvalues of X^T X: [1.975 0.052 0.007 0.   ]


### $p=0.01$

In [178]:
config = get_config()
# config = u.filter_config(config)
config.task.n_tasks = 4
config.task.p_ood = 0.01
model, log = train(config)

train_7782ba140bb9bc2068e7b6ee7d1dcd0d already completed
Loaded model from ../results/linear/train_7782ba140bb9bc2068e7b6ee7d1dcd0d/checkpoint.pt


In [179]:
model, train_task = load_model_and_task("train_7782ba140bb9bc2068e7b6ee7d1dcd0d")
model = model.to(config.device)

In [180]:
train_task.batch_size = 2048
hiddens, xdata = compute_task_vectors(config, model, train_task, layer_index=3)

  0%|          | 0/4 [00:00<?, ?it/s]

In [181]:
global_mean = hiddens.mean(dim=(0,2))
task_vectors = hiddens - global_mean.unsqueeze(0).unsqueeze(2)
task_vectors = task_vectors.mean(dim=-2)
token_vectors = hiddens - task_vectors.unsqueeze(-2) - global_mean.unsqueeze(0).unsqueeze(2)

pairwise_cosine_similarity(task_vectors[:,-1])

tensor([[ 1.000, -0.116, -0.379, -0.333],
        [-0.116,  1.000, -0.564, -0.242],
        [-0.379, -0.564,  1.000, -0.324],
        [-0.333, -0.242, -0.324,  1.000]])

In [183]:
pairwise_cosine_similarity(train_task.task_pool.squeeze(-1))

tensor([[ 1.000,  0.639, -0.048,  0.682],
        [ 0.639,  1.000, -0.556,  0.561],
        [-0.048, -0.556,  1.000, -0.180],
        [ 0.682,  0.561, -0.180,  1.000]])

In [132]:
plot_task_vector_variance_with_fit(hiddens)

In [133]:
tvs_diff_means = plot_task_vector_differences(hiddens)

In [134]:
plot_pairwise_task_vector_variance(hiddens)

In [135]:
samplers_eval = {
        get_task_name(task): get_sharded_batch_sampler(task, True)
        for task in train_task.get_default_eval_tasks(**config["eval"])
    }

RE = Ridge(config.task.noise_scale**2 / config.task.task_scale**2)
dMMSE = DiscreteMMSE(config.task.noise_scale, train_task.task_pool)

plot_mse_vs_position(model, samplers_eval, RE, dMMSE, mode="Pretrain", step=1)

In [136]:
plot_mse_vs_position(model, samplers_eval, RE, dMMSE, mode="Latent", step=1)

In [137]:
lambdas, r2_scores = estimate_lambda_with_r2(task_vectors[:,-1], task_vectors)
plot_lambdas(lambdas)

In [144]:
plot_all_relative_errors(train_task, task_vectors, get_dmmse_posterior)

### $p=0.05$

In [184]:
config = get_config()
# config = u.filter_config(config)
config.task.n_tasks = 4
config.task.p_ood = 0.05
model, log = train(config)

train_43e3e3efdc38445b8b255571cb502ac8 already completed
Loaded model from ../results/linear/train_43e3e3efdc38445b8b255571cb502ac8/checkpoint.pt


In [185]:
model, train_task = load_model_and_task("train_43e3e3efdc38445b8b255571cb502ac8")
model = model.to(config.device)
train_task.batch_size = 2048
hiddens, xdata = compute_task_vectors(config, model, train_task, layer_index=3)

  0%|          | 0/4 [00:00<?, ?it/s]

In [186]:
global_mean = hiddens.mean(dim=(0,2))
task_vectors = hiddens - global_mean.unsqueeze(0).unsqueeze(2)
task_vectors = task_vectors.mean(dim=-2)
token_vectors = hiddens - task_vectors.unsqueeze(-2) - global_mean.unsqueeze(0).unsqueeze(2)

pairwise_cosine_similarity(task_vectors[:,-1])

tensor([[ 1.000, -0.060, -0.464, -0.214],
        [-0.060,  1.000, -0.502, -0.284],
        [-0.464, -0.502,  1.000, -0.403],
        [-0.214, -0.284, -0.403,  1.000]])

In [155]:
plot_task_vector_variance_with_fit(hiddens, normalize=False)

In [156]:
tvs_diff_means = plot_task_vector_differences(hiddens)

In [157]:
plot_pairwise_task_vector_variance(hiddens)

In [158]:
samplers_eval = {
        get_task_name(task): get_sharded_batch_sampler(task, True)
        for task in train_task.get_default_eval_tasks(**config["eval"])
    }

RE = Ridge(config.task.noise_scale**2 / config.task.task_scale**2)
dMMSE = DiscreteMMSE(config.task.noise_scale, train_task.task_pool)

plot_mse_vs_position(model, samplers_eval, RE, dMMSE, mode="Pretrain", step=1)

In [159]:
plot_mse_vs_position(model, samplers_eval, RE, dMMSE, mode="Latent", step=1)

In [160]:
lambdas, r2_scores = estimate_lambda_with_r2(task_vectors[:,-1], task_vectors)
plot_lambdas(lambdas)

In [162]:
plot_all_relative_errors(train_task, task_vectors, get_dmmse_posterior)

### Other settings

In [89]:
config = get_config()
# config = u.filter_config(config)
config.task.n_tasks = 2**7
model, log = train(config)

train_bc1f7727fc8c8c5c6a412115ae4d24c5 already completed
Loaded model from ../results/linear/train_bc1f7727fc8c8c5c6a412115ae4d24c5/checkpoint.pt


In [97]:
model, train_task = load_model_and_task("train_bc1f7727fc8c8c5c6a412115ae4d24c5")
model = model.to(config.device)
train_task.batch_size = 1024
task_vectors = compute_task_vectors(config, model, train_task, layer_index=1)

  0%|          | 0/128 [00:00<?, ?it/s]

In [98]:
plot_task_vector_variance_with_fit(task_vectors)

In [92]:
tvs_diff_means = plot_task_vector_differences(task_vectors)

In [93]:
task_vectors = compute_task_vectors(config, model, train_task, layer_index=2)

  0%|          | 0/128 [00:00<?, ?it/s]

In [96]:
plot_task_vector_variance_with_fit(task_vectors)

In [95]:
tvs_diff_means = plot_task_vector_differences(task_vectors)

In [101]:
plot_pairwise_task_vector_variance(task_vectors)

In [99]:
samplers_eval = {
        get_task_name(task): get_sharded_batch_sampler(task)
        for task in train_task.get_default_eval_tasks(**config["eval"])
    }

RE = Ridge(config.task.noise_scale**2 / config.task.task_scale**2)
dMMSE = DiscreteMMSE(config.task.noise_scale, train_task.task_pool)

plot_mse_vs_position(model, samplers_eval, RE, dMMSE, mode="Pretrain", step=1)

In [100]:
plot_mse_vs_position(model, samplers_eval, RE, dMMSE, mode="Latent", step=1)

In [102]:
tvs_means = task_vectors.mean(dim=-2)
tvs_mean_weighted = tvs_means[:, -1:].mean(dim=1)
lambdas = estimate_lambda(tvs_mean_weighted, tvs_means[:10], reg=1e-3)
plot_lambdas(lambdas)

In [104]:
X = tvs_mean_weighted.numpy()
eigenvalues = np.linalg.eigvals(X @ X.T)
print("Eigenvalues of X^T X:", eigenvalues)

Eigenvalues of X^T X: [1642.259    0.519    0.041    0.017    0.01     0.006    0.006    0.004
    0.003    0.002    0.       0.       0.       0.       0.       0.
    0.       0.       0.       0.       0.      -0.      -0.      -0.
    0.       0.       0.      -0.      -0.      -0.      -0.      -0.
   -0.       0.       0.       0.       0.       0.      -0.      -0.
    0.       0.      -0.      -0.       0.       0.       0.      -0.
   -0.      -0.       0.      -0.      -0.       0.       0.       0.
   -0.      -0.       0.      -0.      -0.       0.      -0.       0.
   -0.       0.       0.       0.      -0.      -0.      -0.      -0.
    0.       0.      -0.       0.      -0.      -0.      -0.       0.
    0.      -0.      -0.       0.       0.       0.      -0.      -0.
   -0.      -0.      -0.       0.       0.       0.      -0.       0.
   -0.      -0.      -0.       0.       0.       0.       0.      -0.
   -0.      -0.      -0.      -0.       0.      -0.      -0.     

### 2**8

In [105]:
config = get_config()
# config = u.filter_config(config)
config.task.n_tasks = 2**8
model, log = train(config)

train_21add59d65446de57febaab2b406b583 already completed
Loaded model from ../results/linear/train_21add59d65446de57febaab2b406b583/checkpoint.pt


In [116]:
model, train_task = load_model_and_task("train_21add59d65446de57febaab2b406b583")
model = model.to(config.device)
train_task.batch_size = 1024
task_vectors = compute_task_vectors(config, model, train_task, layer_index=1)

  0%|          | 0/256 [00:00<?, ?it/s]

In [107]:
plot_task_vector_variance_with_fit(task_vectors)

In [108]:
tvs_diff_means = plot_task_vector_differences(task_vectors)

In [117]:
tvs_means = task_vectors.mean(dim=-2)
tvs_mean_weighted = tvs_means[:, -1:].mean(dim=1)
lambdas = estimate_lambda(tvs_mean_weighted, tvs_means[:10], reg=1e-3)
plot_lambdas(lambdas)

/mnt/c/Users/User/LLM/ICL/.venv/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning:

Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.196e-03, tolerance: 1.829e-03



In [118]:
X = tvs_mean_weighted.numpy()
eigenvalues = np.linalg.eigvals(X @ X.T)
print("Eigenvalues of X^T X:", eigenvalues)

Eigenvalues of X^T X: [4488.731   21.52     0.332    0.013    0.009    0.006    0.005    0.004
    0.004    0.003    0.001    0.       0.       0.       0.      -0.
    0.       0.       0.       0.       0.       0.       0.       0.
    0.       0.      -0.      -0.      -0.      -0.      -0.      -0.
   -0.      -0.      -0.      -0.       0.       0.       0.       0.
    0.       0.       0.       0.      -0.      -0.      -0.      -0.
   -0.      -0.      -0.      -0.       0.       0.       0.       0.
    0.      -0.      -0.      -0.      -0.      -0.      -0.       0.
    0.       0.       0.       0.       0.       0.       0.      -0.
   -0.      -0.      -0.      -0.      -0.      -0.      -0.      -0.
   -0.      -0.      -0.      -0.      -0.      -0.      -0.      -0.
    0.       0.       0.       0.       0.       0.       0.       0.
    0.       0.       0.       0.      -0.      -0.       0.       0.
    0.      -0.      -0.      -0.      -0.       0.       0.     

In [110]:
task_vectors = compute_task_vectors(config, model, train_task, layer_index=2)

  0%|          | 0/256 [00:00<?, ?it/s]

In [111]:
plot_task_vector_variance_with_fit(task_vectors)

In [112]:
tvs_diff_means = plot_task_vector_differences(task_vectors)

In [114]:
plot_pairwise_task_vector_variance(task_vectors)

In [113]:
tvs_means = task_vectors.mean(dim=-2)
tvs_mean_weighted = tvs_means[:, -1:].mean(dim=1)
lambdas = estimate_lambda(tvs_mean_weighted, tvs_means[:10], reg=1e-3)
plot_lambdas(lambdas)

In [109]:
samplers_eval = {
        get_task_name(task): get_sharded_batch_sampler(task)
        for task in train_task.get_default_eval_tasks(**config["eval"])
    }

RE = Ridge(config.task.noise_scale**2 / config.task.task_scale**2)
dMMSE = DiscreteMMSE(config.task.noise_scale, train_task.task_pool)

plot_mse_vs_position(model, samplers_eval, RE, dMMSE, mode="Pretrain", step=1)

In [115]:
X = tvs_mean_weighted.numpy()
eigenvalues = np.linalg.eigvals(X @ X.T)
print("Eigenvalues of X^T X:", eigenvalues)

Eigenvalues of X^T X: [3455.875  105.469   49.272   47.531   44.324   37.312   35.816   32.149
    0.621    0.05     0.012    0.01     0.006    0.005    0.003    0.003
    0.002    0.002    0.001    0.001    0.001    0.001    0.       0.
    0.       0.       0.       0.       0.       0.       0.       0.
    0.       0.       0.       0.       0.       0.       0.       0.
    0.      -0.      -0.       0.       0.       0.       0.       0.
    0.       0.       0.       0.       0.       0.       0.       0.
   -0.      -0.      -0.      -0.      -0.      -0.      -0.      -0.
   -0.      -0.      -0.      -0.      -0.      -0.      -0.      -0.
    0.       0.       0.       0.       0.       0.      -0.      -0.
   -0.      -0.      -0.      -0.      -0.       0.       0.       0.
    0.       0.       0.       0.       0.      -0.      -0.      -0.
   -0.      -0.      -0.      -0.      -0.      -0.      -0.      -0.
   -0.      -0.      -0.       0.       0.       0.       0.  

In [120]:
config = get_config()
# config = u.filter_config(config)
config.task.n_tasks = 2**6
model, log = train(config)

train_aef1b6d75d7d90d6dfd4bca8ceed7888 already completed
Loaded model from ../results/linear/train_aef1b6d75d7d90d6dfd4bca8ceed7888/checkpoint.pt


### 2**6 

In [121]:
model, train_task = load_model_and_task("train_aef1b6d75d7d90d6dfd4bca8ceed7888")
model = model.to(config.device)
train_task.batch_size = 1024
task_vectors = compute_task_vectors(config, model, train_task, layer_index=1)

  0%|          | 0/64 [00:00<?, ?it/s]

In [122]:
plot_task_vector_variance_with_fit(task_vectors)

In [123]:
tvs_diff_means = plot_task_vector_differences(task_vectors)

In [124]:
samplers_eval = {
        get_task_name(task): get_sharded_batch_sampler(task)
        for task in train_task.get_default_eval_tasks(**config["eval"])
    }

RE = Ridge(config.task.noise_scale**2 / config.task.task_scale**2)
dMMSE = DiscreteMMSE(config.task.noise_scale, train_task.task_pool)

plot_mse_vs_position(model, samplers_eval, RE, dMMSE, mode="Pretrain", step=1)

In [125]:
plot_mse_vs_position(model, samplers_eval, RE, dMMSE, mode="Latent", step=1)

In [126]:
tvs_means = task_vectors.mean(dim=-2)
tvs_mean_weighted = tvs_means[:, -1:].mean(dim=1)
lambdas = estimate_lambda(tvs_mean_weighted, tvs_means[:10], reg=1e-3)
plot_lambdas(lambdas)

In [127]:
task_vectors = compute_task_vectors(config, model, train_task, layer_index=2)

  0%|          | 0/64 [00:00<?, ?it/s]